# บทที่ 8 · RustFS + Iceberg: ที่เก็บข้อมูลของเรา (Lakehouse)

session 2 เราใช้ **RustFS** เก็บไฟล์แบบ S3 มาแล้ว บทนี้จะใช้ Python คุยกับ RustFS และสร้าง **ตาราง Iceberg** บนนั้น

บทนี้: **boto3 (bucket / ไฟล์) · Iceberg คืออะไร · table bucket · ต่อ catalog ด้วย DuckDB · namespace bronze/silver/gold ·
สร้าง/เพิ่ม/แก้/ลบ/MERGE · ย้อนเวลา (snapshot) · ดูไฟล์จริงใน bucket**

ก่อนเริ่ม: `docker compose ps` ต้องเห็น `de4-rustfs` เป็น `healthy` · หน้าเว็บ RustFS: http://localhost:9001 (user `student` / `classroom`)

## 8.1 คุยกับ RustFS ด้วย boto3 (ทบทวน session 2 แต่ใช้โค้ดแทนหน้าเว็บ)

In [ ]:
import os

import boto3
from dotenv import load_dotenv

load_dotenv()
s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("RUSTFS_ENDPOINT", "http://127.0.0.1:9000"),
    aws_access_key_id=os.getenv("RUSTFS_ACCESS_KEY", "student"),
    aws_secret_access_key=os.getenv("RUSTFS_SECRET_KEY", "classroom"),
    region_name="us-east-1",
)

print([b["Name"] for b in s3.list_buckets()["Buckets"]])        # bucket ที่มีอยู่

In [ ]:
# สร้าง bucket ชื่อ landing = "จุดรับของ" ไฟล์ที่คนอื่นส่งมาให้เราจะมาวางที่นี่
existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if "landing" not in existing:
    s3.create_bucket(Bucket="landing")

# อัปโหลดไฟล์จากเครื่องเรา -> bucket
s3.upload_file("../data/marketplace/marketplace_orders_2026-09-25.csv", "landing", "demo/orders_2026-09-25.csv")

# ดูว่ามีไฟล์อะไรบ้าง
for obj in s3.list_objects_v2(Bucket="landing", Prefix="demo/").get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes", obj["LastModified"])

In [ ]:
# อ่านไฟล์จาก bucket มาเป็นตารางโดยไม่ต้องดาวน์โหลดลงเครื่อง
import io

import pandas as pd

file_bytes = s3.get_object(Bucket="landing", Key="demo/orders_2026-09-25.csv")["Body"].read()
orders = pd.read_csv(io.BytesIO(file_bytes))
orders.head(3)

In [ ]:
# S3 ไม่มีคำสั่ง "ย้าย" : ต้อง copy ไปที่ใหม่ แล้วลบของเดิม
s3.copy_object(Bucket="landing", Key="done/orders_2026-09-25.csv",
               CopySource={"Bucket": "landing", "Key": "demo/orders_2026-09-25.csv"})
s3.delete_object(Bucket="landing", Key="demo/orders_2026-09-25.csv")
print([o["Key"] for o in s3.list_objects_v2(Bucket="landing").get("Contents", [])])
s3.delete_object(Bucket="landing", Key="done/orders_2026-09-25.csv")          # ลบทิ้ง (แค่ตัวอย่าง)

## 8.2 Iceberg คืออะไร

ไฟล์ CSV/parquet ใน bucket เป็นแค่ "กองกระดาษ" ถ้าอยากแก้ 1 แถว, อยากรู้ว่าตารางมีไฟล์ไหนบ้าง, อยากย้อนดูข้อมูลเมื่อวาน ทำได้ยาก
**Apache Iceberg** ทำให้ไฟล์ใน bucket ใช้งานได้ **เหมือนตารางใน database**

เปรียบเหมือนห้องสมุด:

| Iceberg | ห้องสมุด |
|---|---|
| **data files** (parquet) | หนังสือบนชั้น (ข้อมูลจริง) |
| **metadata** | สารบัญ: ตารางนี้ประกอบด้วยไฟล์ไหนบ้าง มีคอลัมน์อะไร และเก็บสารบัญทุกเวอร์ชันไว้ (ย้อนเวลาได้) |
| **catalog** | บรรณารักษ์: รู้ว่าตารางชื่อนี้ สารบัญเล่มล่าสุดอยู่ที่ไหน |

**RustFS 1.0 มี catalog (แบบ Iceberg REST catalog) อยู่ในตัว** ที่ `http://127.0.0.1:9000/iceberg` ไม่ต้องติดตั้งอะไรเพิ่ม

ส่วนคนที่ "อ่าน/เขียน" ตารางให้เรา คือ **DuckDB** (SQL engine ตัวเล็กที่ติดตั้งผ่าน pip) เราสั่งด้วย SQL ที่เรียนใน session 3 ได้เลย

## 8.3 เตรียม table bucket

bucket ที่จะเก็บตาราง Iceberg ต้อง **เปิดโหมด table bucket** ก่อน (ทำครั้งเดียว)
ใช้ function `make_table_bucket` ในไฟล์ `lakehouse.py` (อยู่โฟลเดอร์เดียวกับ notebook นี้ เปิดอ่านได้)

In [ ]:
from lakehouse import connect_lakehouse, make_table_bucket, show_tables

make_table_bucket("lakehouse")          # สร้าง bucket "lakehouse" + เปิดโหมด table bucket (รันซ้ำได้)
print("พร้อมแล้ว")

## 8.4 ต่อ DuckDB เข้ากับ catalog

`connect_lakehouse()` ทำ 3 อย่าง (ดูโค้ดเต็มได้ใน cell ถัดไป):
1. เปิด DuckDB และโหลดส่วนเสริม `httpfs` (คุยกับ S3) กับ `iceberg`
2. สร้าง **secret** = กุญแจเข้า RustFS
3. **ATTACH** catalog ของ RustFS ตั้งชื่อว่า `lake` → ตารางจะอยู่ในรูป `lake.<namespace>.<ชื่อตาราง>`

In [ ]:
import inspect

import lakehouse

print(inspect.getsource(lakehouse.connect_lakehouse))

In [ ]:
lake = connect_lakehouse()          # ครั้งแรกจะดาวน์โหลดส่วนเสริมของ DuckDB (ต้องต่อเน็ต) รอสักครู่
show_tables(lake)

## 8.5 Namespace: bronze / silver / gold

namespace = "โฟลเดอร์" ของตาราง (เหมือน database ใน MySQL) เราแบ่งตามความสะอาดของข้อมูล เรียกว่า **Medallion**

| ชั้น | เปรียบเหมือน | เก็บอะไร |
|---|---|---|
| **bronze** | ผักที่เพิ่งซื้อจากตลาด | ข้อมูลดิบตามที่ได้มา + บันทึกว่ามาจากไหน เมื่อไหร่ (ไม่แก้ ไม่ลบ) |
| **silver** | ผักล้างและหั่นแล้ว | แปลงชนิดข้อมูลให้ถูก ตัดข้อมูลซ้ำ ทำเครื่องหมายแถวที่เสีย |
| **gold** | อาหารจานที่เสิร์ฟ | ตัวเลขสรุปพร้อมใช้ เช่น ยอดขายรายวัน |

In [ ]:
for layer in ["bronze", "silver", "gold", "demo"]:
    lake.sql(f"CREATE SCHEMA IF NOT EXISTS lake.{layer}")

lake.sql("SELECT schema_name FROM information_schema.schemata WHERE catalog_name = 'lake' ORDER BY 1")

## 8.6 สร้างตารางจาก DataFrame

DuckDB อ่านตัวแปร DataFrame ใน Python ได้ตรง ๆ ใช้ชื่อตัวแปรใน SQL ได้เลย

⚠️ **ข้อจำกัดที่เจอจริง (DuckDB 1.5.5 + Iceberg):** ถ้า **กรอง (WHERE) ด้วยคอลัมน์ข้อความที่มีภาษาไทยยาว ๆ** อาจเจอ error
`Invalid unicode (byte sequence mismatch)` ในตาราง Iceberg ให้ใช้ค่าภาษาอังกฤษ หรือกรองด้วยตัวเลข (id) แทน
(อ่านข้อมูลภาษาไทยได้ปกติ)

In [ ]:
fruit_prices = pd.DataFrame({
    "fruit_id": [1, 2, 3],
    "fruit": ["mango", "durian", "mangosteen"],
    "price_per_kg": [45.0, 250.0, 80.0],
})

lake.sql("DROP TABLE IF EXISTS lake.demo.fruit_prices")
lake.sql("CREATE TABLE lake.demo.fruit_prices AS SELECT * FROM fruit_prices")     # fruit_prices = ตัวแปร DataFrame
lake.sql("SELECT * FROM lake.demo.fruit_prices")

💡 หลังรัน `CREATE TABLE ... AS SELECT` จะเห็นโฟลเดอร์ `data` ว่าง ๆ โผล่ข้าง notebook นี้ (DuckDB สร้างไว้เอง)
**ไม่ต้องสนใจ ลบได้** ข้อมูลจริงอยู่ใน RustFS (ดูในหัวข้อ 8.10)

In [ ]:
# ผลจาก SQL -> DataFrame ใช้ .df()
prices = lake.sql("SELECT * FROM lake.demo.fruit_prices ORDER BY price_per_kg DESC").df()
print(type(prices))
prices

## 8.7 เพิ่ม / แก้ / ลบ (SQL เดียวกับ session 3)

In [ ]:
lake.sql("INSERT INTO lake.demo.fruit_prices VALUES (4, 'rambutan', 60.0), (5, 'longan', 70.0)")
lake.sql("UPDATE lake.demo.fruit_prices SET price_per_kg = 300 WHERE fruit_id = 2")      # ทุเรียนขึ้นราคา
lake.sql("DELETE FROM lake.demo.fruit_prices WHERE fruit_id = 5")                        # ลำไยหมด
lake.sql("SELECT * FROM lake.demo.fruit_prices ORDER BY fruit_id")

In [ ]:
# เพิ่มทีละหลายแถวจาก DataFrame: INSERT ... SELECT
more_fruits = pd.DataFrame({"fruit_id": [6, 7], "fruit": ["banana", "papaya"], "price_per_kg": [30.0, 25.0]})
lake.sql("INSERT INTO lake.demo.fruit_prices SELECT * FROM more_fruits")
lake.sql("SELECT count(*) AS fruits FROM lake.demo.fruit_prices")

## 8.8 MERGE: "มีแล้วให้แก้ ยังไม่มีให้เพิ่ม" (upsert)

ใช้แทน `INSERT ... ON DUPLICATE KEY UPDATE` ของ MySQL (session 3) หัวใจของการโหลดข้อมูลซ้ำได้โดยไม่เกิดแถวซ้ำ

In [ ]:
todays_prices = pd.DataFrame({
    "fruit_id": [1, 3, 8],
    "fruit": ["mango", "mangosteen", "coconut"],
    "price_per_kg": [50.0, 90.0, 35.0],            # มะม่วง/มังคุดราคาเปลี่ยน, มะพร้าวเป็นของใหม่
})

lake.sql("""
    MERGE INTO lake.demo.fruit_prices AS old
    USING todays_prices AS new
    ON old.fruit_id = new.fruit_id
    WHEN MATCHED THEN UPDATE SET price_per_kg = new.price_per_kg      -- มีแล้ว: แก้ราคา
    WHEN NOT MATCHED THEN INSERT *                                    -- ยังไม่มี: เพิ่มแถวใหม่
""")
lake.sql("SELECT * FROM lake.demo.fruit_prices ORDER BY fruit_id")

In [ ]:
# รัน MERGE เดิมซ้ำอีกรอบ -> จำนวนแถวไม่เพิ่ม (ปลอดภัยเวลาต้องรันใหม่)
lake.sql("""
    MERGE INTO lake.demo.fruit_prices AS old
    USING todays_prices AS new
    ON old.fruit_id = new.fruit_id
    WHEN MATCHED THEN UPDATE SET price_per_kg = new.price_per_kg
    WHEN NOT MATCHED THEN INSERT *
""")
lake.sql("SELECT count(*) AS fruits FROM lake.demo.fruit_prices")

## 8.9 Snapshot: ทุกการเปลี่ยนแปลงถูกเก็บเป็นเวอร์ชัน ย้อนดูได้

ทุก INSERT / UPDATE / DELETE / MERGE สร้าง **snapshot** ใหม่ ข้อมูลเก่าไม่หาย (เหมือนประวัติการแก้ไขใน Google Docs)

In [ ]:
history = lake.sql("""
    SELECT sequence_number, snapshot_id, timestamp_ms
    FROM iceberg_snapshots(lake.demo.fruit_prices)
    ORDER BY sequence_number
""").df()
history

In [ ]:
first_version = int(history["snapshot_id"].iloc[0])
print("ตารางตอนสร้างครั้งแรก:")
lake.sql(f"SELECT * FROM lake.demo.fruit_prices AT (VERSION => {first_version}) ORDER BY fruit_id")

💡 ใช้ตอน "เมื่อวานตัวเลขยังถูกอยู่เลย" ย้อนไปดูข้อมูลก่อนที่มีคนแก้ได้

## 8.10 ดูไฟล์จริงใน bucket

ตาราง Iceberg ก็คือไฟล์ธรรมดาใน RustFS: **metadata** (`.json`, `.avro`) + **data** (`.parquet`)

RustFS ไม่ได้ตั้งชื่อโฟลเดอร์ตามชื่อตาราง แต่ใช้ **รหัส** เช่น `tables/10f9b51b-.../`
**catalog** (บรรณารักษ์) คือคนที่รู้ว่า `demo.fruit_prices` อยู่โฟลเดอร์ไหน ถามได้ด้วย `iceberg_metadata()`

In [ ]:
table_files = lake.sql("SELECT content, file_path, record_count FROM iceberg_metadata(lake.demo.fruit_prices)").df()
table_files

In [ ]:
table_folder = table_files["file_path"].iloc[0].split("/data/")[0]          # s3://lakehouse/tables/<รหัส>
prefix = table_folder.replace("s3://lakehouse/", "") + "/"
print("ตารางนี้อยู่ที่:", table_folder)

for f in s3.list_objects_v2(Bucket="lakehouse", Prefix=prefix).get("Contents", []):
    print(f"{f['Size']:>7} bytes  {f['Key'].replace(prefix, '')}")

In [ ]:
# ไฟล์ parquet เปิดอ่านตรง ๆ ได้ด้วย (เป็นแค่ไฟล์ธรรมดา)
first_file = table_files["file_path"].iloc[0]
lake.sql(f"SELECT * FROM read_parquet('{first_file}')")

สังเกต:
- `data/` มีไฟล์ parquet หลายไฟล์ เพราะเราแก้ข้อมูลหลายรอบ ไฟล์เก่ายังอยู่ (ย้อนเวลาได้เพราะแบบนี้)
- การแก้/ลบ ไม่ได้เขียนทับไฟล์เดิม แต่เขียนไฟล์ใหม่ (`POSITION_DELETES` = บันทึกว่าแถวไหนถูกลบ) แล้ว metadata ชี้ไปที่ชุดล่าสุด
- ลองเปิดหน้าเว็บ http://localhost:9001 → bucket `lakehouse` → `tables/` ดูโฟลเดอร์เดียวกันนี้

## 8.11 ลบตารางตัวอย่าง

In [ ]:
lake.sql("DROP TABLE lake.demo.fruit_prices")
show_tables(lake)

## ✅ สรุปบทที่ 8

| เรื่อง | คำสั่ง |
|---|---|
| bucket / ไฟล์ | `s3.create_bucket`, `upload_file`, `list_objects_v2`, `get_object`, `copy_object` + `delete_object` (= ย้าย) |
| เตรียม lakehouse | `make_table_bucket("lakehouse")`, `lake = connect_lakehouse()` |
| namespace | `CREATE SCHEMA IF NOT EXISTS lake.bronze` |
| สร้างตาราง | `CREATE TABLE lake.x.t AS SELECT * FROM my_dataframe` |
| เพิ่ม / แก้ / ลบ | `INSERT`, `UPDATE`, `DELETE` |
| upsert | `MERGE INTO ... USING ... ON ... WHEN MATCHED ... WHEN NOT MATCHED ...` |
| ผลลัพธ์เป็นตาราง pandas | `lake.sql("SELECT ...").df()` |
| ย้อนเวลา | `iceberg_snapshots(...)`, `SELECT ... AT (VERSION => id)` |

🧪 **แบบฝึกท้ายบท:** สร้างตาราง `lake.demo.my_expenses` จาก DataFrame รายจ่าย (id, item เป็นภาษาอังกฤษ, amount) 3 แถว
จากนั้น MERGE ข้อมูลใหม่ 2 แถว (แก้ 1 เพิ่ม 1) แล้วดูว่ามีกี่ snapshot สุดท้าย DROP ตารางทิ้ง

In [ ]:
# เขียนต่อตรงนี้